# 🐈‍⬛⚡ Shadow Cat Relay EdgeFill v27 — Big Iteration

Thank you to the Kaggle community and the competition organizers.

A deterministic, competition-scoped replay-calibrated attack search. This version preserves the proven single-hop EXFIL anchor while adding a two-stage template race, rolling failover, bounded replay-edge fill, and a fast SDK path resolver.

In [ ]:
import glob, os, sys
from pathlib import Path

sys.argv = [sys.argv[0]]
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)
ATTACK_PATH = WORKING_DIR / 'attack.py'
SUBMISSION_PATH = WORKING_DIR / 'submission.csv'

dataset_root = None
for candidate in glob.glob('/kaggle/input/*/kaggle_evaluation') + glob.glob('/kaggle/input/*/*/kaggle_evaluation'):
    parent = str(Path(candidate).parent)
    if (Path(parent) / 'aicomp_sdk').exists():
        dataset_root = parent
        if parent not in sys.path:
            sys.path.insert(0, parent)
        break
print(f'🐈‍⬛ SDK root: {dataset_root or "resolved by attack.py"}')


In [ ]:
import base64, hashlib, os

ATTACK_PY_B64 = '...'
payload = base64.b64decode(ATTACK_PY_B64)
temp_path = ATTACK_PATH.with_suffix('.py.tmp')
temp_path.write_bytes(payload)
os.replace(temp_path, ATTACK_PATH)
print(f'⚡ attack.py written: {ATTACK_PATH.stat().st_size} bytes')
print(f'🐈‍⬛ SHA256: {hashlib.sha256(ATTACK_PATH.read_bytes()).hexdigest()}')


In [ ]:
import ast, importlib.util, py_compile

EXPECTED_SHA256 = '01c337488be94fdf4819b3946280b0128cd569178ac020fec276dfe7db7c9cf2'
source = ATTACK_PATH.read_text(encoding='utf-8')
assert __import__('hashlib').sha256(source.encode('utf-8')).hexdigest() == EXPECTED_SHA256
py_compile.compile(str(ATTACK_PATH), doraise=True)
tree = ast.parse(source)
classes = {node.name: node for node in tree.body if isinstance(node, ast.ClassDef)}
assert 'AttackAlgorithm' in classes
methods = {node.name: node for node in classes['AttackAlgorithm'].body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))}
assert 'run' in methods
assert [arg.arg for arg in methods['run'].args.args][:3] == ['self', 'env', 'config']

spec = importlib.util.spec_from_file_location('shadow_cat_edgefill', ATTACK_PATH)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
smoke = module.AttackAlgorithm({'n_candidates': 3}).run(None, None)
assert len(smoke) == 3
assert len({tuple(getattr(c, 'user_messages', ())) for c in smoke}) == 3
print('FORMAT_GATE | PASS | syntax, contract, deterministic smoke')


In [ ]:
import csv, glob, os, sys

def env_flag(name):
    value = os.getenv(name, '').strip().lower()
    return value not in {'', '0', 'false', 'no', 'off'}

if env_flag('KAGGLE_IS_COMPETITION_RERUN'):
    if SUBMISSION_PATH.exists():
        SUBMISSION_PATH.unlink()
    for root in ['/kaggle/input/ai-agent-security-multi-step-tool-attacks', *sorted(glob.glob('/kaggle/input/*'))]:
        if os.path.isdir(os.path.join(root, 'kaggle_evaluation')):
            if root not in sys.path:
                sys.path.insert(0, root)
            break
    from kaggle_evaluation.jed_attack_134815.jed_attack_inference_server import JEDAttackInferenceServer
    print('🐈‍⬛ MODE: scored competition rerun')
    print('⚡ STRATEGY: two-stage race + replay-safe edge fill + rolling failover')
    JEDAttackInferenceServer().serve()
else:
    rows = [('gpt_oss_public', 0.0), ('gpt_oss_private', 0.0), ('gemma_public', 0.0), ('gemma_private', 0.0)]
    temp_csv = SUBMISSION_PATH.with_suffix('.csv.tmp')
    with temp_csv.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.writer(handle, lineterminator='\n')
        writer.writerow(['Id', 'Score'])
        writer.writerows(rows)
    os.replace(temp_csv, SUBMISSION_PATH)
    print('🐈‍⬛ Shadow Cat Relay EdgeFill v27')
    print('⚡ FINAL STATUS: VALIDATED — READY TO SUBMIT')
